# Memory Native — compute-bottleneck study (A100)

Reproduces `research/results/compute_bottleneck_2026_10_17/`:

1. bit-level verification of the packed counter update (decimation contract, one-update
   per step, no-op at lr=0, oracle parity);
2. full-step cost and memory decomposition for 6 layer shapes x 5 token counts x 6
   implementation variants;
3. the repo's real char-LM training gate: dense AdamW vs qat vs counter vs counter_rms,
   2 seeds, plus a time-to-quality curve and the peak-memory gate.

Reference run: A100-SXM4-40GB, torch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0, commit
`ab39b3f`. Read `REPORT.md` for the analysis. Needs a GPU runtime.

Known issue the verifier reports: on 4096^2 the fused updater is not a strict no-op at
lr=0 (2 codes of 16.8M move by one quantum). That is expected output, not a cell failure.

In [ ]:
import os, subprocess, sys, torch
print(subprocess.run(["bash","-lc","nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
try:
    import triton; print("triton", triton.__version__)
except Exception as e:
    print("triton missing:", e)

In [ ]:
COMMIT = "ab39b3f3b13fc663f58730e25eaf214f6ec17570"
print(subprocess.run(["bash","-lc",f"""
set -e
cd /content
[ -d memory-native ] || git clone -q https://github.com/kharkilirov1/memory-native.git
cd memory-native
git fetch -q origin research/lut-multi-token || true
git checkout -q {COMMIT}
git log --oneline -1
python -m pip install -q -e . 2>&1 | tail -1
"""], capture_output=True, text=True).stdout)

In [ ]:
# 1) bit-level update verification (~1 min; expects the known 4096^2 no-op defect)
print(subprocess.run(["bash","-lc","cd /content/memory-native && python scripts/verify_counter_semantics.py 2>&1 | tail -14"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 2) full-step cost + memory decomposition (~15-25 min; writes JSON/CSV next to the report)
print(subprocess.run(["bash","-lc","cd /content/memory-native && python scripts/measure_step_breakdown.py --iters 15 --reps 3 2>&1 | tail -8"],
                     capture_output=True, text=True).stdout[-4000:])

In [ ]:
# 3) real training gate: same arch/data/budget, seeds 0 and 1 (~2-3 min on A100)
for seed in (0, 1):
    print(subprocess.run(["bash","-lc",f"""
cd /content/memory-native && memory-native-charlm --config tiny --steps 600 --batch 32 \
  --eval-iters 20 --kinds dense,qat,counter,counter_rms --device cuda --seed {seed} 2>&1 \
  | grep -E 'final val|corpus|====' """], capture_output=True, text=True).stdout)

In [ ]:
# 4) time-to-quality curve (separate runs at 150/300/450 steps) + peak-memory gate
for steps in (150, 300, 450):
    print(subprocess.run(["bash","-lc",f"""
cd /content/memory-native && memory-native-charlm --config tiny --steps {steps} --batch 32 \
  --eval-iters 20 --kinds dense,counter_rms --device cuda --seed 0 2>&1 | grep 'final val' """],
                         capture_output=True, text=True).stdout)
print(subprocess.run(["bash","-lc","cd /content/memory-native && memory-native-memgate --config tiny --batch 16 --device cuda --optimizers adamw --seed 0 2>&1 | grep -E 'peak|persistent'"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 5) read back the committed/machine-written measurements
import json, pathlib, collections
OUT = pathlib.Path('/content/memory-native/research/results/compute_bottleneck_2026_10_17')
print('artifacts:', sorted(p.name for p in OUT.glob('*')))
if (OUT/'step_breakdown.json').exists():
    rows = json.load(open(OUT/'step_breakdown.json'))['rows']
    by = collections.defaultdict(dict)
    for r in rows: by[(r['out_f'], r['in_f'], r['tokens'])][(r['variant'], r['phase'])] = r
    kinds = ['dense_fp32', 'packed_fused', 'packed_decode_once', 'packed_cache_g', 'packed_cache_g_dec8']
    print(f"{'shape':>10} {'tok':>5} " + ' '.join(f'{k[-14:]:>15}' for k in kinds))
    for key in sorted(by, key=lambda k: (k[0]*k[1], k[2])):
        if key[2] not in (512, 1024): continue
        vals = []
        for k in kinds:
            r = by[key].get((k, 'full_step'))
            vals.append(f"{r['ms']:15.3f}" if r else f"{'-':>15}")
        print(f"{key[0]:5d}x{key[1]:<5d}{key[2]:>5} " + ' '.join(vals))

## What to expect

* The verifier should print 10 checks and one failure at 4096^2 only: the fused updater
  moves 2 of 16.8M codes with lr=0. The decimation contract and the one-update-per-step
  checks must all pass; the oracle parity row reports ~0.15% code disagreement (TF32
  correlation in the fused kernel), which is the documented quanta-level contract.
* The decomposition should reproduce: fused decode-in-GEMM is 3.6-7.8x slower than
  cuBLAS + one transient decode; decode-once without a persistent cache peaks at 144 MB
  (2.25x denser than dense fp32); the cached group-scope path with dec8 is ~0.87x of
  AdamW at 4096^2/B=512 with 17 MB peak.
* The training gate: counter_rms should reach equal-or-better val loss than dense AdamW
  at 600 steps with ~3.7x less persistent state, but at ~4x lower tokens/s, i.e. ~2x
  slower to equal quality. That is the current bottleneck, documented in REPORT.md.

Everything is measured on the machine that runs the notebook; do not compare absolute
milliseconds across different GPUs, compare ratios.